# 23 — When the table is too big: function approximation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/023-function-approximation.ipynb)

A Q-table has one row per state, so the table grows with every dimension you add to the state. In this notebook you count that growth, replace the table with four weights, and watch those four weights fit a curve over a continuous state. Then you break the model on purpose, twice.

## 1. The numbers: rows you would need

A grid with ten bins per dimension is already a lot of rows, and the count multiplies with each dimension. Read the log column: it is the number of zeros in the row count.

In [ ]:
import numpy as np                                   # the array library
bins = 10                                            # ten bins per dimension, the usual grid
dims = np.array([1, 2, 3, 6])                        # four ways of describing a state
for d in dims:                                       # one printed line per dimension count
    rows = bins ** d                                 # rows needed = 10 to the power d
    print(f"{d} dims -> {rows:>10,} rows, log10 {np.log10(rows):.1f}")   # count and its log
phi = np.array([1., 0.4, 0., 0.])                    # features for state 0.4, action 0
print("features:", phi)                              # always four numbers
print("stored:", phi.size, "against", bins ** 6)     # 4 against a million

**Line by line**

- `import numpy as np` — loads the array library.
- `bins = 10` — how finely each dimension is cut.
- `dims = ...` — the four state descriptions we compare.
- `for d in dims:` — one line of output per dimension count.
- `rows = bins ** d` — the table has one row per cell of the grid.
- `print(f"...")` — the row count and its base-10 log.
- `phi = ...` — one feature row: bias, state, action, and their product.
- `print("features:", phi)` — the model's whole input, four numbers.
- `print("stored:", ...)` — four numbers replace a million-row table.

## 2. Step by step: fitting four weights

The model is `Q(s, a) = w0 + w1*s + w2*a + w3*s*a`, a straight-line fit in each action. We build the true values on a grid of states, then take 200 small steps downhill on the squared error. Watch the error and the weights in the printed checkpoints.

In [ ]:
states = np.linspace(0, 1, 11)                      # eleven states spread over the interval
actions = np.array([0., 1.])                         # two actions, written as numbers
X = []                                               # one feature row per (state, action)
for s in states:                                     # walk the states
    for a in actions:                                # walk the actions
        X.append([1., s, a, s * a])                  # bias, state, action, their product
X = np.array(X)                                      # shape (22, 4)
Y = []                                               # the true value of each row
for s in states:                                     # same states, same order
    for a in actions:                                # same actions, same order
        Y.append(s if a == 0 else 1. - s)            # action 0 pays s, action 1 pays 1 - s
Y = np.array(Y)                                      # shape (22,)
w = np.zeros(4)                                      # start with every weight at zero
lr = 0.3                                             # the step size
for step in range(200):                              # 200 sweeps over the same 22 rows
    pred = X @ w                                     # the current estimate for every row
    grad = X.T @ (pred - Y) / len(Y)                 # gradient of the mean squared error
    w = w - lr * grad                                # one step downhill
    if step % 50 == 0:                               # four checkpoints only
        mse = np.mean((X @ w - Y) ** 2)              # error right now
        print(f"step {step:3d}  mse {mse:.5f}  w {np.round(w, 2)}")   # watch it fall

**Line by line**

- `states = np.linspace(0, 1, 11)` — eleven points cover the continuous state.
- `actions = ...` — two actions, coded as the numbers 0 and 1.
- `X = []` — will hold one row of features per state-action pair.
- `for s in states:` — outer loop over states.
- `for a in actions:` — inner loop over actions.
- `X.append([1., s, a, s * a])` — the four features; `s * a` lets each action bend.
- `X = np.array(X)` — 22 rows of four numbers.
- `Y = []` — the targets, built in exactly the same order.
- `for s in states:` — the same states again.
- `for a in actions:` — the same two actions again.
- `Y.append(s if a == 0 else 1. - s)` — the true value of each pair.
- `Y = np.array(Y)` — a matching column of 22 numbers.
- `w = np.zeros(4)` — the model starts knowing nothing.
- `lr = 0.3` — how far each step moves.
- `for step in range(200):` — the update loop.
- `pred = X @ w` — every prediction at once, as one matrix-vector product.
- `grad = X.T @ (pred - Y) / len(Y)` — slope of the mean squared error.
- `w = w - lr * grad` — move against the slope.
- `if step % 50 == 0:` — print only every fiftieth step.
- `mse = ...` — the error at this checkpoint.
- `print(...)` — error and weights together.

## 3. Plot: the fitted curve and the table's growth

The left panel puts the learned curve on top of the truth for both actions. The right panel returns to section 1 with both axes on a log scale, where a table's growth is a straight line and its slope is the number of dimensions.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
grid = np.linspace(0, 1, 101)                        # a fine grid for smooth curves
q0 = w[0] + w[1] * grid                              # Q(s, 0): bias and state weights only
q1 = q0 + w[2] + w[3] * grid                         # Q(s, 1): the action weights are added
fig, ax = plt.subplots(1, 2, figsize=(9, 3))         # two panels side by side
ax[0].plot(grid, grid, "k--", linewidth=1, label="true Q(s, 0)")      # truth, action 0
ax[0].plot(grid, 1 - grid, "r:", linewidth=1, label="true Q(s, 1)")   # truth, action 1
ax[0].plot(grid, q0, label="learned Q(s, 0)")        # fitted curve, action 0
ax[0].plot(grid, q1, label="learned Q(s, 1)")        # fitted curve, action 1
ax[0].set_xlabel("state s")                          # x-axis label
ax[0].set_ylabel("value Q(s, a)")                    # y-axis label
ax[0].legend(fontsize=7)                             # name the four lines
bins = np.array([2, 4, 8, 16, 32])                   # resolution per dimension
for d in [2, 3, 5]:                                  # three dimension counts
    ax[1].plot(bins, bins ** d, marker="o", label=f"{d} dims")   # rows = bins to the power d
ax[1].set_xscale("log")                              # log x
ax[1].set_yscale("log")                              # log y turns the exponent into a slope
ax[1].set_xlabel("bins per dimension (log)")         # x-axis label
ax[1].set_ylabel("table rows (log)")                 # y-axis label
ax[1].legend(fontsize=7)                             # name the three lines
fig.suptitle("four weights fit a curve; a table grows with every dimension")   # takeaway
plt.tight_layout()                                   # stop the labels overlapping
plt.show()                                           # render

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `grid = ...` — 101 states for a smooth curve.
- `q0 = ...` — Q for action 0, using only two of the four weights.
- `q1 = ...` — Q for action 1; the other two weights shift and tilt it.
- `fig, ax = plt.subplots(1, 2, ...)` — one figure, two panels.
- `ax[0].plot(grid, grid, "k--", ...)` — the true action-0 value, a straight line.
- `ax[0].plot(grid, 1 - grid, ...)` — the true action-1 value.
- `ax[0].plot(grid, q0, ...)` — what the model learned for action 0.
- `ax[0].plot(grid, q1, ...)` — what it learned for action 1.
- `ax[0].set_xlabel(...)` — names the horizontal axis.
- `ax[0].set_ylabel(...)` — names the vertical axis.
- `ax[0].legend(fontsize=7)` — a key for the four lines.
- `bins = ...` — five grid resolutions.
- `for d in [2, 3, 5]:` — one line per dimension count.
- `ax[1].plot(bins, bins ** d, ...)` — rows against resolution.
- `ax[1].set_xscale("log")` — log x.
- `ax[1].set_yscale("log")` — log y; straight lines mean exponential growth.
- `ax[1].set_xlabel(...)` — names the horizontal axis.
- `ax[1].set_ylabel(...)` — names the vertical axis.
- `ax[1].legend(fontsize=7)` — a key for the three lines.
- `fig.suptitle(...)` — the sentence the whole figure is making.
- `plt.tight_layout()` — keeps the two panels from colliding.
- `plt.show()` — render.

## 4. Interactive: drag the step size

The slider sets `lr`, the distance travelled per step, and redraws the error curve on a log scale. Small steps crawl; large steps dive and then bounce around the answer. The title prints the error at the last sweep.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function
def fit(lr=0.3):                                     # lr comes from the slider
    ww = np.zeros(4)                                 # fresh weights on every drag
    hist = []                                        # error after each sweep
    for step in range(200):                          # the same 200 sweeps
        ww = ww - lr * X.T @ (X @ ww - Y) / len(Y)   # one gradient step
        hist.append(np.mean((X @ ww - Y) ** 2))      # record the error
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(hist, color="tab:blue")                 # error against sweep number
    plt.yscale("log")                                # log y keeps the tail visible
    plt.xlabel("sweep")                              # x-axis label
    plt.ylabel("mean squared error (log)")           # y-axis label
    plt.title(f"lr = {lr:.2f}: final error {hist[-1]:.4f}")   # live readout
    plt.show()                                       # draw it
    print(f"lr {lr:.2f} -> final mse {hist[-1]:.5f}")   # the same number as text
interact(fit, lr=(0.01, 1.0, 0.01))                  # attach the slider

**Line by line**

- `from ipywidgets import interact` — the slider widget.
- `def fit(lr=0.3):` — one function per slider position.
- `ww = np.zeros(4)` — every drag starts from zero weights.
- `hist = []` — the error curve for this setting.
- `for step in range(200):` — the same 200 sweeps as section 2.
- `ww = ww - lr * ...` — one step, sized by the slider.
- `hist.append(...)` — store the error after the step.
- `plt.figure(...)` — a new figure for each drag.
- `plt.plot(hist, ...)` — the learning curve.
- `plt.yscale("log")` — the interesting part is the tail.
- `plt.xlabel("sweep")` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.title(...)` — the final error, printed on the plot.
- `plt.show()` — draw it.
- `print(...)` — the same number as text.
- `interact(fit, lr=(0.01, 1.0, 0.01))` — the slider, its range and its step.

## 5. When it breaks: two states, one pair of weights

Now the model sees a coarse bin instead of the exact state, so 0.1 and 0.2 arrive as the same row of features. Their targets are +1 and -1, and one shared weight cannot average those two numbers into anything that fits either. Alternate the updates and watch the two errors take turns being wrong.

In [ ]:
def features(s):                                     # a coarse-coded view of the state
    return np.array([1., np.floor(s / 0.5)])         # half-unit bins: 0.1 and 0.2 share bin 0
print("s=0.1 ->", features(0.1))                     # the row the model sees for 0.1
print("s=0.2 ->", features(0.2))                     # the identical row for 0.2
target = {0.1: 1.0, 0.2: -1.0}                       # opposite targets for the two states
w = np.zeros(2)                                      # one shared weight vector
lr = 0.5                                             # step size
e1, e2 = [], []                                      # error history at each state
for step in range(40):                               # 40 alternating updates
    s = 0.1 if step % 2 == 0 else 0.2                # which state we train on this step
    phi = features(s)                                # identical features either way
    err = phi @ w - target[s]                        # signed error on this state
    w = w - lr * err * phi                           # one least-mean-squares step
    e1.append(abs(features(0.1) @ w - 1.0))          # damage left at state 0.1
    e2.append(abs(features(0.2) @ w + 1.0))          # damage left at state 0.2
plt.figure(figsize=(6, 3))                           # new figure
plt.plot(e1, label="error at s = 0.1 (target +1)")   # first track
plt.plot(e2, label="error at s = 0.2 (target -1)")   # second track
plt.ylim(0, 2.2)                                     # both errors on one scale
plt.xlabel("update number")                          # x-axis label
plt.ylabel("absolute error")                         # y-axis label
plt.legend(fontsize=8)                               # name the two tracks
plt.title("fixing one state breaks the other, forever")   # takeaway
plt.show()                                           # render
print(f"last errors: {e1[-1]:.2f} and {e2[-1]:.2f}")  # neither ever reaches zero

**Line by line**

- `def features(s):` — the model's whole view of a state.
- `return np.array([1., np.floor(s / 0.5)])` — bin number instead of the exact state.
- `print("s=0.1 ->", ...)` — the first state's features.
- `print("s=0.2 ->", ...)` — identical, so nothing can separate them.
- `target = {...}` — one state wants +1, the other wants -1.
- `w = np.zeros(2)` — a single shared weight vector.
- `lr = 0.5` — step size.
- `e1, e2 = [], []` — two error histories.
- `for step in range(40):` — 40 updates.
- `s = 0.1 if step % 2 == 0 else 0.2` — alternate the state.
- `phi = features(s)` — its features.
- `err = phi @ w - target[s]` — prediction minus target.
- `w = w - lr * err * phi` — move the shared weights toward this state.
- `e1.append(...)` — how far state 0.1 now is from +1.
- `e2.append(...)` — how far state 0.2 now is from -1.
- `plt.figure(...)` — a new figure.
- `plt.plot(e1, ...)` — the first error track.
- `plt.plot(e2, ...)` — the second.
- `plt.ylim(0, 2.2)` — both tracks on one readable scale.
- `plt.xlabel("update number")` — names the horizontal axis.
- `plt.ylabel("absolute error")` — names the vertical axis.
- `plt.legend(fontsize=8)` — name the two tracks.
- `plt.title(...)` — the sentence the figure is making.
- `plt.show()` — render.
- `print(f"last errors ...")` — the residual error at the end.

## Your turn

Change the bin width in `features` to 0.05 and predict what happens to the two error tracks before you rerun. Then set `lr` in section 4 to 1.5 and describe the curve you get.